# 🚀 سامانه جامع پیش‌پردازش متون بازیابی اطلاعات (IR System - v2)
### پروژه درس مبانی بازیابی اطلاعات و جستجوی وب
---
**ویژگی‌های نسخه ۲ (v2):**
- **ورودی تعاملی (`input()`):** امکان ورود هر متن دلخواه جدید توسط کاربر و مشاهده گزارش گام‌به‌گام پیش‌پردازش
- **خط‌لوله استاندارد:** توکن‌سازی (`Tokenization`) ➔ یکسان‌سازی (`Normalization`) ➔ حذف کلمات ایست (`Stopwords Removal`) ➔ ریشه‌یابی (`Stemming` با الگوریتم PorterStemmer)
- **ارزیابی بنچمارک:** محاسبه دقیق Precision و Recall برای متون اصلی پروژه در کنار متون جدید کاربر
- **سازگاری کامل:** قابل اجرا در محیط **Google Colab** و **VSCode**

### سلول ۱: کتابخانه‌ها و تنظیمات اولیه (Stopwords & Setup)

In [1]:
import re
import json

# لیست استاندارد کلمات ایست انگلیسی (مستقل و بدون نیاز به اینترنت)
ENGLISH_STOPWORDS = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd",
    'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers',
    'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which',
    'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been',
    'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if',
    'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between',
    'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out',
    'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why',
    'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not',
    'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', "don't", 'should',
    "should've", 'now', 'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', "aren't", 'couldn', "couldn't",
    'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't",
    'ma', 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't",
    'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"
}

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer
    try:
        nltk.download('stopwords', quiet=True)
        STOP_WORDS = set(stopwords.words('english'))
    except Exception:
        STOP_WORDS = ENGLISH_STOPWORDS
except ImportError:
    from nltk.stem import PorterStemmer
    STOP_WORDS = ENGLISH_STOPWORDS

print("✓ تنظیمات اولیه و کلمات ایست با موفقیت بارگذاری شدند.")

✓ تنظیمات اولیه و کلمات ایست با موفقیت بارگذاری شدند.


### سلول ۲: تعریف کلاس موتور پیش‌پردازش (`TextPreprocessor`)

In [2]:
class TextPreprocessor:
    def __init__(self, stop_words_set=None):
        self.stop_words = stop_words_set if stop_words_set is not None else STOP_WORDS
        self.stemmer = PorterStemmer()

    def tokenize(self, text: str) -> list:
        # 1. حذف پسوند مالکیت ('s)
        text = re.sub(r"['’]s\b", "", text, flags=re.IGNORECASE)
        # 2. الگوی استخراج مخفف‌ها، کلمات مرکب و کلمات عادی
        pattern = r"(?:[A-Za-z]\.){2,}|[A-Za-z]+(?:-[A-Za-z]+)+|[A-Za-z0-9]+"
        return re.findall(pattern, text)

    def normalize(self, tokens: list) -> list:
        normalized = []
        for token in tokens:
            tok = token.lower()
            if re.match(r"^(?:[a-z]\.){2,}$", tok):
                tok = tok.replace(".", "")
            normalized.append(tok)
        return normalized

    def remove_stopwords(self, tokens: list) -> list:
        return [tok for tok in tokens if tok not in self.stop_words]

    def stem(self, tokens: list) -> list:
        return [self.stemmer.stem(tok) for tok in tokens]

    def process_with_details(self, text: str) -> dict:
        tokens = self.tokenize(text)
        normalized = self.normalize(tokens)
        no_stopwords = self.remove_stopwords(normalized)
        stemmed_terms = self.stem(no_stopwords)

        return {
            "raw_text": text,
            "1_tokens": tokens,
            "2_normalized": normalized,
            "3_no_stopwords": no_stopwords,
            "4_stemmed_terms": stemmed_terms,
            "unique_terms": sorted(list(set(stemmed_terms)))
        }

    def process(self, text: str) -> list:
        return self.process_with_details(text)["4_stemmed_terms"]

preprocessor = TextPreprocessor()
print("✓ کلاس TextPreprocessor آماده استفاده است.")

✓ کلاس TextPreprocessor آماده استفاده است.


### 🎯 سلول ۳: پیش‌پردازش تعاملی متون جدید (ورودی دلخواه کاربر)
شما می‌توانید هر متن جدید انگلیسی را در کادر زیر وارد کنید و دکمه **Run** را بزنید تا گزارش کامل مراحل پیش‌پردازش را مشاهده کنید.

In [3]:
# @title 📝 کادر ورودی متن جدید (ادیت کنید و اجرا کنید)
user_text = "Artificial intelligence is transforming state-of-the-art search engines like Google in the U.S.A.!" # @param {type:"string"}

# اجرای خط‌لوله پیش‌پردازش
result = preprocessor.process_with_details(user_text)

print("=" * 60)
print("📄 متن ورودی کاربر:", result["raw_text"])
print("=" * 60)
print("1️⃣ توکن‌سازی (Tokenization):       ", result["1_tokens"])
print("2️⃣ یکسان‌سازی (Normalization):    ", result["2_normalized"])
print("3️⃣ حذف کلمات ایست (Stopwords):   ", result["3_no_stopwords"])
print("4️⃣ ریشه‌یابی (Stemming):          ", result["4_stemmed_terms"])
print("-" * 60)
print("✨ توالی نهایی اصطلاحات خروجی:   ", result["4_stemmed_terms"])
print("✨ مجموعه متمایز (Vocabulary):     ", result["unique_terms"])
print("=" * 60)

📄 متن ورودی کاربر: Artificial intelligence is transforming state-of-the-art search engines like Google in the U.S.A.!
1️⃣ توکن‌سازی (Tokenization):        ['Artificial', 'intelligence', 'is', 'transforming', 'state-of-the-art', 'search', 'engines', 'like', 'Google', 'in', 'the', 'U.S.A.']
2️⃣ یکسان‌سازی (Normalization):     ['artificial', 'intelligence', 'is', 'transforming', 'state-of-the-art', 'search', 'engines', 'like', 'google', 'in', 'the', 'usa']
3️⃣ حذف کلمات ایست (Stopwords):    ['artificial', 'intelligence', 'transforming', 'state-of-the-art', 'search', 'engines', 'like', 'google', 'usa']
4️⃣ ریشه‌یابی (Stemming):           ['artifici', 'intellig', 'transform', 'state-of-the-art', 'search', 'engin', 'like', 'googl', 'usa']
------------------------------------------------------------
✨ توالی نهایی اصطلاحات خروجی:    ['artifici', 'intellig', 'transform', 'state-of-the-art', 'search', 'engin', 'like', 'googl', 'usa']
✨ مجموعه متمایز (Vocabulary):      ['artifici', 'engin', 'goog

### 💬 سلول ۴: دریافت تعاملی متن با `input()` همزمان (پیمایش مداوم)
این سلول از کاربر ورودی زنده می‌گیرد تا چندین جمله جدید را پشت سر هم تست کنید.

In [4]:
print("برای خروج عبارت 'exit' یا 'q' را وارد کنید.\n")
while True:
    text_in = input("📝 متن جدید را وارد کنید: ").strip()
    if not text_in:
        continue
    if text_in.lower() in ['exit', 'q', 'quit']:
        print("پایان حالت تعاملی.")
        break

    res = preprocessor.process_with_details(text_in)
    print(f"\n➔ توالی اصطلاحات: {res['4_stemmed_terms']}")
    print(f"➔ واژگان متمایز:  {res['unique_terms']}\n" + "-"*50)

برای خروج عبارت 'exit' یا 'q' را وارد کنید.

📝 متن جدید را وارد کنید: today is a rainy day 

➔ توالی اصطلاحات: ['today', 'raini', 'day']
➔ واژگان متمایز:  ['day', 'raini', 'today']
--------------------------------------------------
📝 متن جدید را وارد کنید: q
پایان حالت تعاملی.


### 📊 سلول ۵: ارزیابی بنچمارک ۳ متن اصلی پروژه و محاسبه Precision & Recall

In [5]:
def evaluate_output(program_output: list, gold_standard: set) -> dict:
    O = set(program_output)
    G = set(gold_standard)
    intersection = O.intersection(G)
    precision = len(intersection) / len(O) if len(O) > 0 else 0.0
    recall = len(intersection) / len(G) if len(G) > 0 else 0.0
    return {
        "O": sorted(list(O)),
        "G": sorted(list(G)),
        "intersection": sorted(list(intersection)),
        "precision": precision,
        "recall": recall,
        "extra_terms": sorted(list(O - G)),
        "missed_terms": sorted(list(G - O))
    }

SAMPLE_TEXTS = {
    "text1": "Friends, Romans, countrymen, lend me your ears;",
    "text2": "John's new U.S.A. laptop is a state-of-the-art device. The price is high, but the quality is amazing!",
    "text3": "The authorization for this authorized project is pending. He authorizes the payments regularly."
}

GOLD_STANDARDS = {
    "text1": {"friend", "roman", "countrymen", "lend", "ear"},
    "text2": {"john", "new", "usa", "laptop", "state-of-the-art", "devic", "price", "high", "qualiti", "amaz"},
    "text3": {"author", "project", "pend", "payment", "regularli"}
}

final_outputs = {}
print("=" * 66)
print("          ارزیابی بنچمارک متون اصلی پروژه بازیابی اطلاعات          ")
print("=" * 66 + "\n")

for text_id, raw_text in SAMPLE_TEXTS.items():
    terms_seq = preprocessor.process(raw_text)
    final_outputs[text_id] = terms_seq
    eval_res = evaluate_output(terms_seq, GOLD_STANDARDS[text_id])

    print(f"--- [ {text_id} ] ---")
    print(f"متن اصلی: {raw_text}")
    print(f"اصطلاحات خروجی: {terms_seq}")
    print(f"Precision: {eval_res['precision']:.4f} | Recall: {eval_res['recall']:.4f}")
    print("-" * 66 + "\n")

# ذخیره نتایج در outputs.json
with open("outputs.json", "w", encoding="utf-8") as f:
    json.dump(final_outputs, f, ensure_ascii=False, indent=2)
print("✓ فایل 'outputs.json' با موفقیت ذخیره شد.")

          ارزیابی بنچمارک متون اصلی پروژه بازیابی اطلاعات          

--- [ text1 ] ---
متن اصلی: Friends, Romans, countrymen, lend me your ears;
اصطلاحات خروجی: ['friend', 'roman', 'countrymen', 'lend', 'ear']
Precision: 1.0000 | Recall: 1.0000
------------------------------------------------------------------

--- [ text2 ] ---
متن اصلی: John's new U.S.A. laptop is a state-of-the-art device. The price is high, but the quality is amazing!
اصطلاحات خروجی: ['john', 'new', 'usa', 'laptop', 'state-of-the-art', 'devic', 'price', 'high', 'qualiti', 'amaz']
Precision: 1.0000 | Recall: 1.0000
------------------------------------------------------------------

--- [ text3 ] ---
متن اصلی: The authorization for this authorized project is pending. He authorizes the payments regularly.
اصطلاحات خروجی: ['author', 'author', 'project', 'pend', 'author', 'payment', 'regularli']
Precision: 1.0000 | Recall: 1.0000
------------------------------------------------------------------

✓ فایل 'outputs.json' ب